# Radiation observation preprocessing. 

Takes single-point tower observations of SWin at 15-min and 30-min freq and converts to 2min for use in CanRad L2R.

Nic Tarasewicz
2025-07-18

In [ ]:
import pandas as pd
import xarray as xr
import numpy as np
import datetime as dt
import time as tm
import pytz
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import matplotlib.dates as mdates
import plotly.express as px
import seaborn as sns
from scipy.interpolate import griddata
from scipy.interpolate import RectBivariateSpline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.ndimage import gaussian_filter
from sklearn.cluster import KMeans
from datetime import datetime, timedelta
import matplotlib.dates as mdates

In [ ]:
def interpolate_to_frequency(df, freq):
    # Select only numeric columns for processing
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Remove duplicate timestamps
    numeric_df = numeric_df.groupby(numeric_df.index).mean()
    
    # Create a new index with the desired frequency
    new_index = pd.date_range(start=numeric_df.index.min(), end=numeric_df.index.max(), freq=freq)
    interpolated_df = numeric_df.reindex(new_index).interpolate(method='time')
    interpolated_df.index.name = 'Time'
    
    # Add back non-numeric columns (if necessary) with forward fill
    non_numeric_df = df.select_dtypes(exclude=[np.number]).groupby(df.index).first()
    non_numeric_df = non_numeric_df.reindex(new_index).fillna(method='ffill')
    
    # Combine
    final_df = pd.concat([interpolated_df, non_numeric_df], axis=1)
    
    return final_df

In [ ]:
# load EcoTram tower observations

# Load and preprocess MET tower data
def preprocess_met_data(file_path, sheet_name, station_type):
    # Load data
    df = pd.read_excel(file_path, sheet_name=sheet_name, parse_dates=['TIMESTAMP'], index_col='TIMESTAMP')
    df.index.rename('Time', inplace=True)

    # Drop unnecessary columns
    columns_to_drop = [
        'RECORD', 'PTemp_C_Avg', 'BattV_Min', 'SBT_C_Avg', 'WSDiag', 'SmplsF', 
        'Diag1F_Tot', 'Diag2F_Tot', 'Diag4F_Tot', 'Diag8F_Tot', 'Diag9F_Tot', 
        'Diag10F_Tot', 'NNDF'
    ]
    df = df.drop(columns=columns_to_drop)

    # Unit conversions and adjustments
    if 'DBTCDT_Avg' in df.columns:
        df['DBTCDT_Avg'] *= 2.54  # Convert inches to cm
    if 'SHF_Avg' in df.columns and station_type == 'forest':
        df['SHF_Avg'] *= -1  # Invert signal orientation for forest station
    if 'SWin_Avg' in df.columns:
        df['SWin_Avg'] = df['SWin_Avg'].clip(lower=0)  # Clip SWin values below zero to zero

    # Add station identifier
    df['station'] = station_type
    return df

# Define file path
file_path = '/Users/nita1760/Desktop/ch1 manuscript/submission/data/ecotram.xlsx'

for15 = preprocess_met_data(file_path, sheet_name='forest_15min', station_type='forest')
wet15 = preprocess_met_data(file_path, sheet_name='wetland_15min', station_type='wetland')

In [ ]:
# Interpolate to 2-minute intervals
for2 = interpolate_to_frequency(for15, '2min')
wet2 = interpolate_to_frequency(wet15, '2min')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Define time range
date = '2023-03-28'
# date = '2023-07-11'
start_time = f'{date} 06:00'
end_time = f'{date} 21:00'

# Ensure the datetime index is sorted
for15 = for15.sort_index()
wet15 = wet15.sort_index()

# Now slice the sorted data
for15_july9 = for15.loc[start_time:end_time]
wet15_july9 = wet15.loc[start_time:end_time]

# Plotting - resized for half a PowerPoint slide and easy readability
plt.figure(figsize=(6.5, 4.5))
plt.plot(for15_july9.index, for15_july9['SWin_Avg'], label='Forest', linewidth=2)
plt.plot(wet15_july9.index, wet15_july9['SWin_Avg'], label='Wetland', linewidth=2)

plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.xlabel('Hour of Day', fontsize=13)
plt.ylabel('SWin_Avg (W/m²)', fontsize=13)

plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Export 

start_date = '2023-03-28'
end_date = '2023-03-28'

# Filter by date range
forest_filtered = for2.loc[start_date:end_date]

# Select only SWin
forest_swin = forest_filtered[['SWin_Avg']]

# Format the index as 'DD.MM.YYYY HH:MM' and reset index
forest_swin.index = forest_swin.index.strftime('%d.%m.%Y\t%H:%M')
forest_swin.reset_index(inplace=True)

# Export to text file with proper formatting
with open('/Users/nita1760/Desktop/ch1/L2R/met_forest_SWin_2min_v2_rmse.txt', 'w') as file:
    for _, row in forest_swin.iterrows():
        file.write(f"{row[0]}\t{row[1]:.3f}\n")

# Repeat for the wetland data
wetland_filtered = wet2.loc[start_date:end_date]
wetland_swin = wetland_filtered[['SWin_Avg']]
wetland_swin.index = wetland_swin.index.strftime('%d.%m.%Y\t%H:%M')
wetland_swin.reset_index(inplace=True)

with open('/Users/nita1760/Desktop/ch1/L2R/met_wetland_SWin_2min_v2_rmse.txt', 'w') as file:
    for _, row in wetland_swin.iterrows():
        file.write(f"{row[0]}\t{row[1]:.3f}\n")

# for wider run (may 1 - oct 31)
# with open('/Users/nita1760/Desktop/ch1/L2R/met_wetland_SWin_2min_wide_run.txt', 'w') as file:
#     for _, row in wetland_swin.iterrows():
#         file.write(f"{row[0]}\t{row[1]:.3f}\n")

print("Export completed for both forest and wetland data.")

In [ ]:
# US-NR1 2023 comparison runs

# Define column names for NR1 data
column_names = [
    'Year', 'Month', 'Day', 'Hour', 'Minute', 'Sec', 'Decimal Day of Year (MST)',
    'Rsw_in_25m_KZ', 'Rsw_out_25m_KZ', 'Rlw_in_25m_KZ', 'Rlw_out_25m_KZ',
    'Rppfd_in_25m', 'Rppfd_out_25m', 'Rnet_25m_REBS', 'Rnet_0200cm_REBS',
    'Rsw_in_sp510', 'Rsw_in_sn500', 'Rsw_out_sp610', 'Rsw_out_sn500',
    'Rlw_in_sl510', 'Rlw_in_sn500', 'Rlw_out_sn500', 'Tir_si121',
    'T_therm_si121', 'T_therm_sl510', 'T_med_uc', 'snowdepth_judd', 'Rsw_in_sp510_mV',
    'Rsw_out_sp610_mV', 'Rpile_in_sl510_mV', 'T_therm_sl510_mV', 'Tir_pile_si121_mV',
    'Tir_term_si121_mV'
]

# URL for NR1 data
url = 'https://urquell.colorado.edu/data_ameriflux/data_supplemental/niwot_USNR1_radiation_2023_ver.2024.03.01.csv'

# Load
nr1 = pd.read_csv(url, names=column_names, header=None)
nr1['Time'] = pd.to_datetime(nr1[['Year', 'Month', 'Day', 'Hour', 'Minute']])

# Set timezone to UTC-7 to match EcoTram
nr1['Time'] = nr1['Time'].dt.tz_localize('Etc/GMT+7')
nr1.set_index('Time', inplace=True)

# Drop unnecessary columns
columns_to_drop = ['Year', 'Month', 'Day', 'Hour', 'Minute', 'Sec', 'Decimal Day of Year (MST)']
nr1.drop(columns=columns_to_drop, inplace=True)

# Add station identifier
nr1['station'] = 'US-NR1'

# Convert columns to numeric
for col in nr1.columns:
    nr1[col] = pd.to_numeric(nr1[col], errors='coerce')

# Iinterpolate to a specific frequency
def interpolate_to_frequency(df, freq):
    # Select only numeric columns for processing
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Remove duplicate timestamps
    numeric_df = numeric_df.groupby(numeric_df.index).mean()
    new_index = pd.date_range(start=numeric_df.index.min(), end=numeric_df.index.max(), freq=freq, tz='Etc/GMT+7')
    
    # Reindex and interpolate
    interpolated_df = numeric_df.reindex(new_index).interpolate(method='time')
    interpolated_df.index.name = 'Time'
    
    # Add back non-numeric columns
    non_numeric_df = df.select_dtypes(exclude=[np.number]).groupby(df.index).first()
    non_numeric_df = non_numeric_df.reindex(new_index).fillna(method='ffill')
    
    # Combine numeric and non-numeric data
    final_df = pd.concat([interpolated_df, non_numeric_df], axis=1)
    
    return final_df

# Interpolate to 2-minute intervals
nr1_2 = interpolate_to_frequency(nr1, '2min')

# Define start and end dates as timezone-aware timestamps
start_date = pd.Timestamp('2023-03-28', tz='Etc/GMT+7')
end_date = pd.Timestamp('2023-03-29', tz='Etc/GMT+7')

# Filter by date range
nr1_2_filtered = nr1_2.loc[start_date:end_date]

# Select only the Rsw_in_25m_KZ column
nr1_2_swin = nr1_2_filtered[['Rsw_in_25m_KZ']] # top of tower
# nr1_2_swin = nr1_2_filtered[['Rsw_in_sn500']] # 2.2 m above ground

# Convert negative values to zero
nr1_2_swin = nr1_2_swin.clip(lower=0)

# Format the index as 'DD.MM.YYYY HH:MM' and reset index
nr1_2_swin.index = nr1_2_swin.index.strftime('%d.%m.%Y\t%H:%M')
nr1_2_swin.reset_index(inplace=True)

# Export
output_path = '/Users/nita1760/Desktop/ch1/L2R/nr1_SWin_2min_v2_rmse.txt'
with open(output_path, 'w') as file:
    for _, row in nr1_2_swin.iterrows():
        file.write(f"{row[0]}\t{row[1]:.3f}\n")
print("Export complete.")

Export complete.


/var/folders/83/9lmp5tzj3xn0jx_b07zsbf440000gn/T/ipykernel_71944/158431606.py:58: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  non_numeric_df = non_numeric_df.reindex(new_index).fillna(method='ffill')
/var/folders/83/9lmp5tzj3xn0jx_b07zsbf440000gn/T/ipykernel_71944/158431606.py:92: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  file.write(f"{row[0]}\t{row[1]:.3f}\n")
